In [41]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from scipy.stats import chi2_contingency
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support
import warnings
import os

In [42]:
# import the data
df = pd.read_csv('D:/Study/IFRS-9-Complient-Risk-Analysis-modelling/data/missing_handled_dataset.csv')
print(df.shape)

(51215, 82)


### 5.1 Performing Light Feature Selection

In [43]:
# removing column PROSPECTID
df.drop('PROSPECTID' , axis = 1 , inplace=True)
print(df.shape)

(51215, 81)


In [44]:
# ============================================
# STEP 3.3.1 : Basic Feature Audit
# ============================================

feature_audit = pd.DataFrame({
    "Data Type": df.dtypes,
    "Missing Values": df.isnull().sum(),
    "Missing %": (df.isnull().mean() * 100).round(2),
    "Unique Values": df.nunique(),
})

feature_audit

,Data Type,Missing Values,Missing %,Unique Values
Total_TL,int64,0,0.0,107
Tot_Closed_TL,int64,0,0.0,100
Tot_Active_TL,int64,0,0.0,32
Total_TL_opened_L6M,int64,0,0.0,20
Tot_TL_closed_L6M,int64,0,0.0,18
...,...,...,...,...
GL_Flag,int64,0,0.0,2
last_prod_enq2,object,0,0.0,6
first_prod_enq2,object,0,0.0,6
Credit_Score,int64,0,0.0,250


In [45]:
# Checking for constant columns
# ============================================
# Constant Features
# ============================================

constant_cols = feature_audit[
    feature_audit["Unique Values"] == 1
].index.tolist()

print(f"Constant Columns ({len(constant_cols)}):")
print(constant_cols)

Constant Columns (0):
[]


- No constant columns -> GOOD

In [46]:
# ============================================
# Duplicate Columns
# ============================================

duplicate_cols = []

cols = df.columns

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        if df[cols[i]].equals(df[cols[j]]):
            duplicate_cols.append(cols[j])

print(f"Duplicate Columns ({len(duplicate_cols)}):")
print(duplicate_cols)

Duplicate Columns (1):
['pct_of_active_TLs_ever']


In [47]:
df.drop(columns=duplicate_cols, inplace=True)

print(f"Remaining Features : {df.shape[1]}")

Remaining Features : 80


### Near Zero Variance Features
Features where one category dominates almost all observations contribute very little information and can often be removed.

In [48]:
# ============================================
# Near-Zero Variance Features
# ============================================

threshold = 0.99  # 99%

nzv_cols = []

for col in df.columns:
    top_freq = df[col].value_counts(normalize=True, dropna=False).iloc[0]

    if top_freq >= threshold:
        nzv_cols.append({
            "Column": col,
            "Most Frequent Value %": round(top_freq * 100, 2)
        })

nzv_df = pd.DataFrame(nzv_cols)

print(f"Near-Zero Variance Features ({len(nzv_df)})")
nzv_df

Near-Zero Variance Features (8)


,Column,Most Frequent Value %
0,num_sub_6mts,99.90
1,num_sub_12mts,99.69
2,num_dbt,99.67
3,num_dbt_6mts,99.97
4,num_dbt_12mts,99.92
5,num_lss,99.80
6,num_lss_6mts,99.98
7,num_lss_12mts,99.95


- Let us review these 8 columns to check if these contain information

In [49]:
for col in nzv_df['Column']:
    print('='*50)
    print(f'Col -> {col}')
    print('='*50)
    print(df[col].value_counts())

Col -> num_sub_6mts
num_sub_6mts
0    51164
1       18
2       14
4        7
3        6
5        5
8        1
Name: count, dtype: int64
Col -> num_sub_12mts
num_sub_12mts
0     51057
1        61
2        42
3        14
6        11
5         9
4         6
10        5
8         3
11        3
9         1
20        1
12        1
7         1
Name: count, dtype: int64
Col -> num_dbt
num_dbt
0     51045
1        45
3        15
2        12
5        12
6        11
9        10
7         8
8         7
11        6
13        5
10        4
25        4
14        4
24        3
22        3
26        3
15        2
16        2
12        2
17        2
18        2
19        2
4         1
23        1
31        1
32        1
35        1
28        1
Name: count, dtype: int64
Col -> num_dbt_6mts
num_dbt_6mts
0    51199
4        8
3        4
5        2
2        1
6        1
Name: count, dtype: int64
Col -> num_dbt_12mts
num_dbt_12mts
0     51175
2        11
10        6
9         5
3         4
1         4
4     

| Feature         | Dominant Value | Recommendation | Reason                                                                             |
| --------------- | -------------- | -------------- | ---------------------------------------------------------------------------------- |
| `num_sub_6mts`  | 0 (99.95%)     | **Keep**       | Recent loan submissions are rare but highly indicative of credit-seeking behavior. |
| `num_sub_12mts` | 0 (99.74%)     | **Keep**       | Even a few submissions may indicate increased credit demand.                       |
| `num_dbt`       | 0 (99.71%)     | **Keep**       | Debt write-offs/defaults are rare but among the strongest risk indicators.         |
| `num_dbt_6mts`  | 0 (99.98%)     | **Keep**       | Extremely rare, but any occurrence is a strong signal.                             |
| `num_dbt_12mts` | 0 (99.94%)     | **Keep**       | Same reasoning.                                                                    |
| `num_lss`       | 0 (99.84%)     | **Keep**       | Loss-related events are uncommon but very informative.                             |
| `num_lss_6mts`  | 0 (99.98%)     | **Keep**       | Rare adverse event.                                                                |
| `num_lss_12mts` | 0 (99.95%)     | **Keep**       | Same reasoning.                                                                    |


- These are all business important features, we cannot remove them. These are not some generic numerical features.

### 5.2. Statistical Testing For Feature Selection

## Feature Selection

1. **Hypothesis Testing**: \
Are these two are associated: MARITALSTATUS and Approved_Flag ? \
**H0: Null hypothesis :** Not associated \
**H1: Alternative hypothesis :** Associated

2. **alpha** : *Significance level* / *Margin of Error* (how much the estimate can be wrong): Assumed : = **0.05**
- less risky projects : high alpha
- high risky projects : low alpha

3. **Confidence Interval** = (1-**alpha**)

4. Calculate the evidence against H0 : **p-value** 
- calculate using tests : T-test, Chi-Square test, ANOVA based on *degree of freedom*. If:
- **p-value** <= **alpha** : Reject H0
- **p-value** > **alpha** : Fail to Reject H0  
(not saying Accept H0 because we don't have enough evidence of it being True either, we just failed to prove it is False: in actual it may or may not be False. The conclusion is based on our defined confidence interval).

When to use what test :
- **Chi-Square** test: Categorical vs Categorical
- **T-test**: Categorical vs Numerical (2 categories)
- **ANOVA**: Categorical vs Numerical (>2 categories)

### Categorical Feature Selection:
- To determine which categorical features are associated with the Target variable.
- Estimate **Chi-Sqaure** test : get **p-value** for each feature.
- Keep only those features whose **p-value** < **alpha** (0.05).

In [50]:
# Chi-Square test:
for i in ['MARITALSTATUS', 'EDUCATION', 'GENDER', 'last_prod_enq2', 'first_prod_enq2']:
    chi2, pval, _, _ = chi2_contingency(pd.crosstab(df[i], df['Approved_Flag']))
    print(i, '---', pval)       # print p-values

# since all the Categorical features have p-value <= 0.05, we will keep them all.

MARITALSTATUS --- 1.4621892175736472e-256
EDUCATION --- 2.2884395916270033e-37
GENDER --- 0.00021894090411806155
last_prod_enq2 --- 0.0
first_prod_enq2 --- 0.0


### Numeric Features

In [52]:
# check how many columns are Numerical:
numeric_columns = []
for i in df.columns:
    if df[i].dtype != 'object' and i not in ['PROSPECTID','Approved_Flag']:
        numeric_columns.append(i)
print(len(numeric_columns))
numeric_columns


74


['Total_TL',
 'Tot_Closed_TL',
 'Tot_Active_TL',
 'Total_TL_opened_L6M',
 'Tot_TL_closed_L6M',
 'pct_tl_open_L6M',
 'pct_tl_closed_L6M',
 'pct_active_tl',
 'pct_closed_tl',
 'Total_TL_opened_L12M',
 'Tot_TL_closed_L12M',
 'pct_tl_open_L12M',
 'pct_tl_closed_L12M',
 'Tot_Missed_Pmnt',
 'Auto_TL',
 'CC_TL',
 'Consumer_TL',
 'Gold_TL',
 'Home_TL',
 'PL_TL',
 'Secured_TL',
 'Unsecured_TL',
 'Other_TL',
 'Age_Oldest_TL',
 'Age_Newest_TL',
 'time_since_recent_payment',
 'num_times_delinquent',
 'max_recent_level_of_deliq',
 'num_deliq_6mts',
 'num_deliq_12mts',
 'num_deliq_6_12mts',
 'max_deliq_6mts',
 'max_deliq_12mts',
 'num_times_30p_dpd',
 'num_times_60p_dpd',
 'num_std',
 'num_std_6mts',
 'num_std_12mts',
 'num_sub',
 'num_sub_6mts',
 'num_sub_12mts',
 'num_dbt',
 'num_dbt_6mts',
 'num_dbt_12mts',
 'num_lss',
 'num_lss_6mts',
 'num_lss_12mts',
 'recent_level_of_deliq',
 'tot_enq',
 'CC_enq',
 'CC_enq_L6m',
 'CC_enq_L12m',
 'PL_enq',
 'PL_enq_L6m',
 'PL_enq_L12m',
 'time_since_recent_enq

### Numerical Feature Selection:
- To determine which numerical features are associated with the Target variable.
- Test to be done : **ANOVA** (since Target varible: 'Approved_Flag' has > 2 categories).
- But before performing ANOVA, **Multicollinearity** between the feature variables needs to be determined : **VIF (Variance Inflation Factor)**.

#### Multicollinearity vs Correlation
- **Multicollinearity** : When two or more features are high associated. Predictability of each feature by other features.
- **Correlation** : It is specific to linear relationship between columns. 
- When we are not sure of whether the feature should have linear relationship or not (because of lack of domain knowledge) : it's safer to use **VIF** rather than Correlation.

#### **VIF (Variance Inflation Factor)**:
- Used to identify Multicollinearity among independent features.
- VIF<sub>i</sub> = 1 / (1- R<sub>i</sub><sup>2</sup>), where R<sub>i</sub><sup>2</sup> is the *coefficient of determination**.
- if VIF<sub>i</sub> > 6 : Highly Multicollinear with other features : remove that feature variable.

VIF: Sequential vs Parallel method:
- Sequentially - Correct method : one out of the several multicollinear features are kept.
- Parallel - Incorrect method : all of the multicollinear features are dropped.

In [ ]:
# VIF sequentially check

vif_data = df[numeric_columns]          # Numerical Columns
total_columns = vif_data.shape[1]       # no. of numerical cols
columns_to_be_kept = []                 # Numerical Columns to be kept
column_index = 0

for i in range (0, total_columns):
    
    vif_value = variance_inflation_factor(vif_data, column_index)       # VIF of ith feature column
    print (column_index,'---',vif_value)
    
    
    if vif_value <= 6:
        columns_to_be_kept.append( numeric_columns[i] )                 # if VIF <= 6: keep the feature
        column_index = column_index+1
    
    else:
        vif_data = vif_data.drop([ numeric_columns[i] ] , axis=1)       # else drop the feature: (SEQUENTIAL VIF)

len(columns_to_be_kept)         # No. of Numerical Columns to be kept (out of 74) = 43

c:\Users\Pradhuman\anaconda3\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


0 --- inf


c:\Users\Pradhuman\anaconda3\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


0 --- inf
0 --- 10.913341798369874
0 --- 8.29875768329047
0 --- 6.430097483452966
0 --- 5.423730728828856
1 --- 2.4967148800916568
2 --- 2926.6585213013473
2 --- 6.842559408252372
2 --- 8.047463736814569
2 --- 3.712058863060794
3 --- 5.414275357607792
4 --- 4.1730434738182005
5 --- 1.9148747649553188


c:\Users\Pradhuman\anaconda3\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


6 --- inf
6 --- 4.775013854362573
7 --- 21.344086463458552
7 --- 32.51911521539646
7 --- 4.454278277474442
8 --- 3.044617715698285
9 --- 2.7224278806444193
10 --- 4.184911909894028
11 --- 2.1808380436752013
12 --- 4.756286322954607
13 --- 4.34659312932927
14 --- 2.5145610161916494
15 --- 8.441084059873175
15 --- 5.510535073289065


c:\Users\Pradhuman\anaconda3\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


16 --- inf
16 --- 9.222322808247723
16 --- 1.7255949645517905
17 --- 2.540971822838544
18 --- 3.5819127915245934
19 --- 8.560813704011789
19 --- 1.6964905915503905
20 --- 7.063120464658584
20 --- 15.39063101421366
20 --- 1.8618621403382598
21 --- 1.5710356577000029
22 --- 2.547638336769991
23 --- 3.1072855940020756
24 --- 2.199015312836947
25 --- 7.384604764105959
25 --- 2.075374824688351
26 --- 2.725158003702161
27 --- 6.2949173292297305
27 --- 2.710747473043839
28 --- 5.305498976249238
29 --- 16.18327887837093
29 --- 6.386755027109229
29 --- 8.84775278412296
29 --- 2.364713386224004
30 --- 8.609344950153716
30 --- 13.012920202363217
30 --- 3.4479075804684225
31 --- 1.564850948272892
32 --- 17.635855712789777
32 --- 10.563248163240466
32 --- 2.2642191384613524
33 --- 21.78072344054973
33 --- 2.9008854263819095
34 --- 3.3999787786244067
35 --- 7.027336055331351
35 --- 2.0745631956244095
36 --- 3.0993284453196335
37 --- 2.770762212554968
38 --- 20.9996500970002
38 --- 16.60032810854501


43

In [54]:
# check ANOVA for columns_to_be_kept : determine which Numerical Columns are associated with the Target variable 'Approved_Flag'

from scipy.stats import f_oneway

columns_to_be_kept_numerical = []       # Final Numerical Columns to be kept

for i in columns_to_be_kept:
    a = list(df[i])  
    b = list(df['Approved_Flag'])  
    
    # dividing the feature values into four groups for each of the Target variable category:
    group_P1 = [value for value, group in zip(a, b) if group == 'P1']
    group_P2 = [value for value, group in zip(a, b) if group == 'P2']
    group_P3 = [value for value, group in zip(a, b) if group == 'P3']
    group_P4 = [value for value, group in zip(a, b) if group == 'P4']


    f_statistic, p_value = f_oneway(group_P1, group_P2, group_P3, group_P4)     # One-way ANOVA

    if p_value <= 0.05:
        columns_to_be_kept_numerical.append(i)      # keep the feature if the p-value is <= 0.05

In [56]:
len(columns_to_be_kept_numerical) # only 41 numeric features are now left 

41

In [57]:
# listing all the final features:

features = columns_to_be_kept_numerical + ['MARITALSTATUS', 'EDUCATION', 'GENDER', 'last_prod_enq2', 'first_prod_enq2']
df = df[features + ['Approved_Flag']]


In [58]:
df.shape # this is our final selected features after Feature Selection

(51215, 47)

### 5.3. Categorical Columns Encoding

In [59]:
# Categorical columns
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

# Remove target if present
if "Approved_Flag" in cat_cols:
    cat_cols.remove("Approved_Flag")

print(f"Categorical Features ({len(cat_cols)}):")
print(cat_cols)

Categorical Features (5):
['MARITALSTATUS', 'EDUCATION', 'GENDER', 'last_prod_enq2', 'first_prod_enq2']


**Check Cardinality**

In [60]:
cardinality = pd.DataFrame({
    "Feature": cat_cols,
    "Unique Values": [df[col].nunique() for col in cat_cols]
}).sort_values("Unique Values")

cardinality

,Feature,Unique Values
0,MARITALSTATUS,2
2,GENDER,2
3,last_prod_enq2,6
4,first_prod_enq2,6
1,EDUCATION,7


In [61]:
df['EDUCATION'].value_counts()

EDUCATION
GRADUATE          16637
12TH              14432
SSC                9254
UNDER GRADUATE     5481
OTHERS             2909
POST-GRADUATE      2236
PROFESSIONAL        266
Name: count, dtype: int64

| Feature           | Unique Values | Recommended Encoding  | Reason                                |
| ----------------- | ------------: | --------------------- | ------------------------------------- |
| `MARITALSTATUS`   |             2 | Binary Encoding (0/1) | Binary categorical                    |
| `GENDER`          |             2 | Binary Encoding (0/1) | Binary categorical                    |
| `EDUCATION`       |             7 | **Ordinal Encoding**  | Natural educational hierarchy         |
| `last_prod_enq2`  |             6 | **One-Hot Encoding**  | Nominal categories, no inherent order |
| `first_prod_enq2` |             6 | **One-Hot Encoding**  | Nominal categories, no inherent order |


### Binary Encoding

In [62]:
# Binary Encoding

df["GENDER"] = df["GENDER"].map({
    "M": 1,
    "F": 0
})

df["MARITALSTATUS"] = df["MARITALSTATUS"].map({
    "Married": 1,
    "Single": 0
})

### Ordinal Encoding

In [63]:
education_mapping = {
    "OTHERS": -1,
    "SSC": 0,
    "12TH": 1,
    "UNDER GRADUATE": 2,
    "GRADUATE": 3,
    "POST-GRADUATE": 4,
    "PROFESSIONAL": 5
}

df["EDUCATION"] = df["EDUCATION"].map(education_mapping)

### OneHotEncoding

In [64]:
df = pd.get_dummies(
    df,
    columns=["first_prod_enq2", "last_prod_enq2"],
    dtype="int"
)

In [65]:
# final check
print(df.shape)

(51215, 57)


In [66]:
# saving the dataset
df.to_csv('D:/Study/IFRS-9-Complient-Risk-Analysis-modelling/data/encoded_dataset.csv',index=False)

### END OF THE NOTEBOOK -> NEXT MODEL DEVELOPMENT